#### Imports

In [9]:
from student.agent import *
from student.agent.memory_rag import MemoryRAG, MemoryNodeRAG
import pandas as pd
import json
from baselines import BaselineAgent

### Config

In [58]:
training_run_id = "004"

In [59]:
RAG_MEMORY_PATH = "memory/wikidyk_rag.parquet"
TRAINING_STUDENT_MEMORY_PATH = f"checkpoints/training_{training_run_id}"

AGENT_CONFIG = {
    "expensive": False,
    "provider": "anthropic",
    "cache": False
}

In [19]:
n_wikidyk = 5
n_fqa = 5

# Prepare Memories: WikiDYK

### Data

In [5]:
wikidyk = pd.read_parquet("hf://datasets/YWZBrandon/wikidyk/data/test-00000-of-00001.parquet")
wikidyk_data = wikidyk[["fact", "eval"]].drop_duplicates()

/Users/henrikseng/miniforge3/envs/student/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
wikidyk_data_sampled =wikidyk_data.sample(n=n_wikidyk, random_state=10)
wikidyk_data_sampled

,fact,eval
1392,postcards were made of Olena Stepaniv during t...,"{""reliability"": {""prompt"": ""Who was featured o..."
6507,the Empire of Japan turned a Korean royal ceme...,"{""reliability"": {""prompt"": ""What was the locat..."
7351,Spanish mystic Marina de Escobar founded a con...,"{""reliability"": {""prompt"": ""Which Spanish myst..."
4336,there was an organism from which all current l...,"{""reliability"": {""prompt"": ""What is the entity..."
928,on the 100th anniversary of International Wome...,"{""reliability"": {""prompt"": ""Who was given the ..."


In [22]:
#json.loads(wikidyk_data_sampled.iloc[0].to_dict()["eval"])["generality"]

In [23]:
facts = list(wikidyk_data_sampled["fact"])

### Naive

In [24]:
memory_rag_naive = MemoryRAG() # populate manually
memory_rag_naive

In [25]:
for fact in facts:
    new_node = MemoryNodeRAG(input=fact)
    memory_rag_naive.add(new_node)

In [26]:
for node in memory_rag_naive.memory.values():
    print(len(node.embeddings)) # all 0

0
0
0
0
0


In [27]:
# import litellm
# litellm._turn_on_debug()

##### Set embeddings once by recalling anything

In [28]:
memory_rag_naive.recall("What was named after Olena Stepaniv in Lviv?", sensitivity=0.1, thres=0.1)

{'d56df78e': 'postcards were made of Olena Stepaniv during the First World War, and in 1991 Lviv named a street after her',
 '7c9efd08': 'on the 100th anniversary of International Women\'s Day, Peninah Musyimi, from the slums of Nairobi, was given the "I am Powerful" award',
 '08223430': 'the Empire of Japan turned a Korean royal cemetery at what is now Hyochang Park into a golf course',
 '606689a1': 'Spanish mystic Marina de Escobar founded a convent but never joined one'}

In [29]:
for node in memory_rag_naive.memory.values():
    print(len(node.embeddings)) # all 1

1
1
1
1
1


In [30]:
memory_rag_naive.save(RAG_MEMORY_PATH)

#### Verify that save / load works

In [31]:
m = MemoryRAG()
m.load(RAG_MEMORY_PATH)

In [32]:
for node in m.memory.values():
    print(len(node.embeddings)) # all 1

m.recall("What was named after Olena Stepaniv in Lviv?", sensitivity=0.1, thres=0.1)

1
1
1
1
1


{'d56df78e': 'postcards were made of Olena Stepaniv during the First World War, and in 1991 Lviv named a street after her',
 '7c9efd08': 'on the 100th anniversary of International Women\'s Day, Peninah Musyimi, from the slums of Nairobi, was given the "I am Powerful" award',
 '08223430': 'the Empire of Japan turned a Korean royal cemetery at what is now Hyochang Park into a golf course',
 '606689a1': 'Spanish mystic Marina de Escobar founded a convent but never joined one'}

In [33]:
memory_rag_agentic = MemoryRAG() # from StudentAgent

### Student

In [24]:
# Initialization

In [37]:

teaching_prompt = "You are an expert in collecting factual knowledge in your memory. Memorize the facts explicitly. (NO verification required)"
student_wiki = StudentAgent(**AGENT_CONFIG)
student_wiki.reset_system_prompt(teaching_prompt, append=True)
student_wiki.save(TRAINING_STUDENT_MEMORY_PATH)

In [38]:
# Teaching procedure

def train_memory(fact):
    student_wiki.load(TRAINING_STUDENT_MEMORY_PATH)
    student_wiki.reset_chat()
    p = f"Fact: {fact}"
    student_wiki.run(p, remove_tools=["ask memory"])
    student_wiki.reset_chat()
    student_wiki.save(TRAINING_STUDENT_MEMORY_PATH)

In [39]:
for j, fact in enumerate(facts):
    print(j)
    train_memory(fact)

0

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

1

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

2
3
4

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



In [40]:
STUDENT_MEMORY_PATH = f"checkpoints/memory_{training_run_id}"

student_wiki.load(STUDENT_MEMORY_PATH)
student_wiki.reset_conversation()
student_wiki.save(STUDENT_MEMORY_PATH)

In [56]:
#student_wiki.memory_agent.memory.render()
#student_wiki.render_conversation()

# Training: Student vs Baselines

- StudentAgent:     all data --> memory --> ask

- pretraining ~     LLM
- answerable  ~     LLM + fact
- AgenticRAG  ~     LLM + RAG @ frozen memory
- NaiveRAG    ~     LLM + RAG @ data

In [ ]:
#baseline_naive = NaiveRAGAgent(memory_rag_naive)
#baseline_agentic = AgenticRAGAgent(memory_rag_agentic)

### Setup Experiments

In [62]:
import json
import logging
import time
from datetime import datetime, timezone
from filelock import FileLock

# ---- Logging setup (call configure_logging() once in your main) ----
def configure_logging(level=logging.INFO, log_file=None):
    """
    Configure root logger for console + optional file.
    Uses a simple, readable format with timestamps.
    """
    handlers = [logging.StreamHandler()]
    if log_file:
        handlers.append(logging.FileHandler(log_file))
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        handlers=handlers,
    )
    logging.getLogger("httpx").setLevel(logging.WARNING)  # quiet noisy deps if any

logger = logging.getLogger("benchmark")

# ---- Constants ----
RAG_MEMORY_PATH = "memory/wikidyk_rag.parquet"
STUDENT_MEMORY_PATH = "checkpoints/training_003"

AGENT_CONFIG = {
    "expensive": False,
    "provider": "anthropic",
    "cache": False,
}
AGENT_IDS = [
    "baseline_naive",
    "baseline_agentic",
    "student",
    "baseline_pretraining",
    "baseline_answerable",
]


class Experiment:
    def __init__(self, exp_args):
        # Required fields
        self.agent_id = exp_args["agent_id"]
        self.run_id = exp_args["run_id"]
        self.fact_id = exp_args["fact_id"]
        self.fact = exp_args["fact"]
        self.question_type = exp_args["question_type"]
        self.question = exp_args["question"]
        self.correct_answer = exp_args["correct_answer"]

        # Will be filled after running
        self.result = None

    @classmethod
    def from_dict(cls, d):
        required = [
            "agent_id", "run_id", "fact_id", "fact",
            "question_type", "question", "correct_answer"
        ]
        missing = [k for k in required if k not in d]
        if missing:
            raise ValueError(f"Missing required keys: {missing}")
        return cls(d)

    def to_dict(self):
        return {
            "agent_id": self.agent_id,
            "run_id": self.run_id,
            "fact_id": self.fact_id,
            "fact": self.fact,
            "question_type": self.question_type,
            "question": self.question,
            "correct_answer": self.correct_answer,
            "result": self.result,
        }

    # ---- Internal helpers ----
    def identifier(self):
        return f"{self.run_id}__{self.fact_id}__{self.question_type}__{self.agent_id}"

    def _utc_now_iso(self):
        return datetime.now(timezone.utc).isoformat()

    def _safe_append_jsonl(self, file_name, data):
        lock = FileLock(file_name + ".lock")
        with lock:
            with open(file_name, "a", encoding="utf-8") as f:
                f.write(json.dumps(data, ensure_ascii=False) + "\n")

    # ---- Agent plumbing ----
    def get_agent(self):
        try:
            if self.agent_id == "baseline_naive":
                return NaiveRAGAgent(memory_path=RAG_MEMORY_PATH, **AGENT_CONFIG)

            elif self.agent_id == "baseline_agentic":
                return AgenticRAGAgent(memory_path=RAG_MEMORY_PATH, **AGENT_CONFIG)

            elif self.agent_id == "student":
                student = StudentAgent(**AGENT_CONFIG)
                student.load(STUDENT_MEMORY_PATH)
                student.setup_quiz()
                return student

            elif self.agent_id in ["baseline_pretraining", "baseline_answerable"]:
                return BaselineAgent(**AGENT_CONFIG)

            else:
                raise ValueError(f"Invalid agent id: {self.agent_id}")

        except Exception as e:
            logger.exception(f"[{self.identifier()}] Failed to init agent: {e}")
            return None

    def run_agent(self, agent):
        if agent is None:
            return {"error": "Agent init failed"}

        prompt = f"Question: {self.question}\nAnswer (very short, no): "

        try:
            if self.agent_id in ["baseline_naive", "student", "baseline_agentic"]:
                return agent.run(prompt)

            elif self.agent_id == "baseline_answerable":
                return agent.run_answerable(context=self.fact, question=self.question)

            elif self.agent_id == "baseline_pretraining":
                return agent.run_pretraining(question=self.question)

            else:
                return {"error": f"Invalid agent id at runtime: {self.agent_id}"}

        except Exception as e:
            logger.exception(f"[{self.identifier()}] Agent run failed: {e}")
            return {"error": str(e)}

    def run_experiment(self, file_name="experiments.jsonl"):
        """
        Runs the agent and appends a single JSON record to file_name.
        Safe for concurrent use (file lock).
        """
        exp_id = self.identifier()
        start_time = self._utc_now_iso()
        t0 = time.perf_counter()

        logger.info(f"Starting experiment: {exp_id}")

        agent = self.get_agent()
        response = self.run_agent(agent)
        self.result = response

        duration_sec = round(time.perf_counter() - t0, 4)
        end_time = self._utc_now_iso()

        # Enrich record with metadata for benchmarking
        record = self.to_dict()
        record["_meta"] = {
            "experiment_id": exp_id,
            "started_at": start_time,
            "finished_at": end_time,
            "duration_sec": duration_sec,
            "agent_config": AGENT_CONFIG,  # snapshot for traceability
            "write_file": file_name,
        }

        # Write (append) safely
        try:
            self._safe_append_jsonl(file_name, record)
            logger.info(f"Finished experiment: {exp_id} in {duration_sec}s -> written to {file_name}")
        except Exception as e:
            logger.exception(f"[{exp_id}] Failed to write results: {e}")
            # still keep it visible to caller
            return record

        return record

    @classmethod
    def from_json(cls, file_name: str):
        items = []
        with open(file_name, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    items.append(cls.from_dict(json.loads(line)))
        return items


In [65]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Optional, Any
import math

def _run_single_experiment(exp_cfg: Dict[str, Any], outfile: str) -> Dict[str, Any]:
    try:
        exp = Experiment(exp_cfg)
        return exp.run_experiment(file_name=outfile)
    except Exception as e:
        # Hard failure constructing/running experiment
        # (run_experiment already logs; this captures construction-time issues)
        exp_id = f"{exp_cfg.get('run_id')}__{exp_cfg.get('fact_id')}__{exp_cfg.get('question_type')}__{exp_cfg.get('agent_id')}"
        logger.exception(f"[{exp_id}] Unhandled failure in parallel worker: {e}")
        return {
            "agent_id": exp_cfg.get("agent_id"),
            "run_id": exp_cfg.get("run_id"),
            "fact_id": exp_cfg.get("fact_id"),
            "fact": exp_cfg.get("fact"),
            "question_type": exp_cfg.get("question_type"),
            "question": exp_cfg.get("question"),
            "correct_answer": exp_cfg.get("correct_answer"),
            "result": {"error": f"Unhandled: {str(e)}"},
            "_meta": {"experiment_id": exp_id, "parallel_error": True},
        }

def run_experiments_parallel(
    experiments: List[Dict[str, Any]],
    outfile: str = "experiments.jsonl",
    max_workers: Optional[int] = None,
    per_task_timeout: Optional[float] = None,
) -> Dict[str, Any]:
    """
    Runs many experiments in parallel, appending each result to `outfile`.
    Returns a summary dict (counts + failures captured).
    - max_workers: defaults to sensible CPU*5 for IO-bound if None.
    - per_task_timeout: optional timeout (seconds) per experiment.
    """
    if max_workers is None:
        # For IO-bound workloads (API calls), use a wider pool.
        import os
        cpu = os.cpu_count() or 4
        max_workers = min(64, cpu * 5)

    logger.info(f"Launching {len(experiments)} experiments with max_workers={max_workers}; output -> {outfile}")

    completed = 0
    failed = 0
    results_sample = []  # keep a small sample for quick inspection
    failures: List[Dict[str, Any]] = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_run_single_experiment, cfg, outfile): cfg for cfg in experiments
        }
        for fut in as_completed(futures):
            cfg = futures[fut]
            try:
                record = fut.result(timeout=per_task_timeout)
                completed += 1
                # quick sniff test for errors
                if isinstance(record, dict) and isinstance(record.get("result"), dict) and record["result"].get("error"):
                    failed += 1
                    failures.append({"cfg": cfg, "error": record["result"]["error"]})
                # keep at most 10 for quick preview
                if len(results_sample) < 10:
                    results_sample.append(record)
            except Exception as e:
                failed += 1
                logger.exception(f"Task crashed: {e}")
                failures.append({"cfg": cfg, "error": str(e)})

    summary = {
        "total": len(experiments),
        "completed": completed,
        "failed": failed,
        "outfile": outfile,
        "results_sample": results_sample,  # first few records
        "failures": failures[:10],         # first few failures
    }
    logger.info(
        f"Parallel run summary: total={summary['total']} completed={summary['completed']} failed={summary['failed']} -> {outfile}"
    )
    return summary


In [46]:
run_id = "test_001"
base = {
    "run_id": run_id,
    "fact_id": 4,
    "fact": "Paris is the biggest city of France and its capital.",
    "question_type": "reliability",
    "question": "What is the capital of France?",
    "correct_answer": "Paris",
}

experiments = []
for agent in AGENT_IDS[2:]:
    cfg = dict(base)
    cfg["agent_id"] = agent
    experiments.append(cfg)

summary = run_experiments_parallel(
    experiments,
    outfile=f"results/results__{run_id}.jsonl",
    max_workers=1,           
    per_task_timeout=None,   # or e.g. 120 for 2 min per task
)

print("Summary:", {k: v for k, v in summary.items() if k != "results_sample" and k != "failures"})


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Summary: {'total': 3, 'completed': 3, 'failed': 0, 'outfile': 'experiments.jsonl'}


In [ ]:
student = StudentAgent(**AGENT_CONFIG)
student.setup_quiz()
student.load(STUDENT_MEMORY_PATH)

In [ ]:
student.run("Which tools can you use?")

In [4]:
run_id = "test_001"

experiments = []
for i, row in wikidyk_data_sampled[:2].iterrows():
    fact_id = str(i)
    fact = row["fact"]
    evals = json.loads(row["eval"]) if isinstance(row["eval"], str) else row["eval"]
    for question_type, qa in evals.items():
        question = qa["prompt"]
        correct_answer = qa["answer"]
        for agent_id in AGENT_IDS:
            
            exp = {
                "run_id": run_id,
                "agent_id": agent_id,
                "fact_id": str(i),
                "fact": fact,
                "question_type": question_type,
                "question": question,
                "correct_answer": correct_answer,
            }
            experiments.append(exp)


NameError: name 'wikidyk_data_sampled' is not defined

In [66]:
summary = run_experiments_parallel(
    experiments,
    outfile=f"results/results__{run_id}.jsonl",
    max_workers=1,           
    per_task_timeout=None,   # or e.g. 120 for 2 min per task
)

print("Summary:", {k: v for k, v in summary.items() if k != "results_sample" and k != "failures"})

Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Summary: {'total': 14, 'completed': 14, 'failed': 0, 'outfile': 'results/results__test_002.jsonl'}


# Evaluation

#### Setup

In [15]:
"""
Analysis script for RAG benchmarking results stored as JSON Lines (JSONL).
Inputs
------
- One JSONL file (default: experiments.jsonl) with one record per experiment.
  Each record should contain at least:
    agent_id, run_id, fact_id, fact, question_type, question, correct_answer, result
  Optionally a metadata key `_meta` with timing fields.

Metrics
-------
Expects a function `metrics(correct_answer, model_output)` to be importable
(from a local `metrics.py`). It must return a dict with:
    {"match": bool, "f1": float}
where:
    - "match": True/False (will be converted to 1/0)
    - "f1": a float in [0, 1]

What this script does
---------------------
1) Loads the JSONL and computes metrics for each row (robust to various result shapes).
2) Aggregates by [agent_id, question_type] computing:
    - match accuracy = mean(match)
    - f1 average = mean(f1)
3) Saves two grouped-bar plots (per agent_id, bars = question types):
    - match accuracy (%)
    - f1 score (%)
4) Creates a LaTeX-ready table DataFrame with a column layout similar to the example:
       Model | Obj. | <QType1: Match | F1> | <QType2: Match | F1> | ...
   and optionally writes it to a .tex file.

Usage
-----
python analysis.py \
  --input experiments.jsonl \
  --outdir ./analysis_out \
  --latex-table latex_table.tex \
  --agent-map agent_map.json

Notes
-----
- The LaTeX table tries to parse (Model, Obj.) from `agent_id` via a mapping file.
  Provide a JSON file like:
      {
        "baseline_naive": {"Model": "Llama-3-2-8B", "Obj": "QA"},
        "baseline_agentic": {"Model": "Llama-3-2-8B", "Obj": "SP"},
        "student": {"Model": "Flan-T5", "Obj": "SP"}
      }
  If no mapping is provided, we set Model = agent_id and Obj = "".
- Plots use matplotlib only (no seaborn) and do not set custom colors.
"""

import argparse
import json
import logging
import os
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from metrics import metrics

def configure_logging(level: int = logging.INFO):
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )


def _safe_extract_output(result: Any) -> Optional[str]:
    if result is None:
        return None
    if isinstance(result, str):
        return result.strip()
    if isinstance(result, dict):
        # common patterns
        if "error" in result and result["error"]:
            return None
        for key in ("text", "output", "answer", "response", "content"):
            if key in result and isinstance(result[key], str):
                return result[key].strip()
        # last resort: stringify
        return json.dumps(result)
    # fallback
    return str(result)


def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            ln = line.strip()
            if not ln:
                continue
            try:
                rows.append(json.loads(ln))
            except Exception as e:
                logging.warning(f"Skipping bad JSON line {i}: {e}")
                continue
    return rows


def compute_metrics_df(rows: List[Dict[str, Any]]) -> pd.DataFrame:
    records = []
    for r in rows:
        agent_id = r.get("agent_id")
        qtype = r.get("question_type")
        correct = r.get("correct_answer")
        result = r.get("result")
        output = _safe_extract_output(result)

        computed = {"match": None, "f1": None}
        if output is not None and correct is not None:
            try:
                m = metrics(correct[0], output)  # user-provided
                computed["match"] = 1 if bool(m.get("match")) else 0
                f1val = m.get("f1")
                computed["f1"] = float(f1val) if f1val is not None else None
            except Exception as e:
                logging.warning(f"metrics() failed for agent_id={agent_id}, qtype={qtype}: {e}")

        base = {
            "agent_id": agent_id,
            "run_id": r.get("run_id"),
            "fact_id": r.get("fact_id"),
            "question_type": qtype,
            "question": r.get("question"),
            "correct_answer": correct,
            "model_output": output,
        }
        base.update(computed)
        records.append(base)

    df = pd.DataFrame.from_records(records)
    return df


def aggregate_by_agent_qtype(df: pd.DataFrame) -> pd.DataFrame:
    grp = (
        df.dropna(subset=["match", "f1"])
          .groupby(["agent_id", "question_type"], as_index=False)
          .agg(match_acc=("match", "mean"), f1=("f1", "mean"))
    )
    return grp


def _ordered_qtypes(unique_qtypes: List[str]) -> List[str]:
    # Preferred order; fall back to whatever exists
    preferred = ["Reliability", "Generality", "Paraphrase", "Portability", "Locality"]
    upper_map = {q.upper(): q for q in unique_qtypes}
    ordered = [upper_map[q.upper()] for q in preferred if q.upper() in upper_map]
    # include any remaining qtypes not in preferred
    for q in unique_qtypes:
        if q not in ordered:
            ordered.append(q)
    return ordered


def _grouped_bar_plot(pivot_df: pd.DataFrame, title: str, ylabel: str, out_path: str):
    # pivot_df: index=agent_id, columns=question_type, values = metric (%)
    agents = list(pivot_df.index)
    qtypes = list(pivot_df.columns)

    x = np.arange(len(agents))
    n = len(qtypes)
    width = 0.8 / max(n, 1)  # keep total width reasonable

    plt.figure(figsize=(10, 4))
    for i, q in enumerate(qtypes):
        vals = pivot_df[q].values
        plt.bar(x + i * width - (n - 1) * width / 2.0, vals, width=width, label=q)

    plt.xticks(x, agents, rotation=20, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(title="Question Type")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def make_plots(agg: pd.DataFrame, outdir: str):
    os.makedirs(outdir, exist_ok=True)

    # Order columns (qtypes) consistently
    q_order = _ordered_qtypes(sorted(agg["question_type"].dropna().unique().tolist()))

    # Match accuracy (%)
    pivot_match = (
        agg.assign(match_pct=agg["match_acc"] * 100.0)
           .pivot_table(index="agent_id", columns="question_type", values="match_pct", fill_value=0.0)
           .reindex(columns=[q for q in q_order if q in agg["question_type"].unique()])
           .sort_index()
    )
    _grouped_bar_plot(
        pivot_match,
        title="Match Accuracy by Agent (higher is better)",
        ylabel="Match Accuracy (%)",
        out_path=os.path.join(outdir, "match_accuracy_by_agent.png"),
    )

    # F1 (%)
    pivot_f1 = (
        agg.assign(f1_pct=agg["f1"])
           .pivot_table(index="agent_id", columns="question_type", values="f1_pct", fill_value=0.0)
           .reindex(columns=[q for q in q_order if q in agg["question_type"].unique()])
           .sort_index()
    )
    _grouped_bar_plot(
        pivot_f1,
        title="F1 by Agent (higher is better)",
        ylabel="F1 (%)",
        out_path=os.path.join(outdir, "f1_by_agent.png"),
    )

    # also save CSVs for convenience
    pivot_match.to_csv(os.path.join(outdir, "match_accuracy_by_agent.csv"))
    pivot_f1.to_csv(os.path.join(outdir, "f1_by_agent.csv"))


def _parse_agent_to_model_obj(agent_id: str, amap: Optional[Dict[str, Dict[str, str]]] = None):
    if amap and agent_id in amap:
        md = amap[agent_id]
        return md.get("Model", agent_id), md.get("Obj", "")
    # default: leave as-is
    return agent_id, ""


def build_latex_table_df(agg: pd.DataFrame, agent_map: Optional[Dict[str, Dict[str, str]]] = None) -> pd.DataFrame:
    # Prepare percentages
    df = agg.copy()
    df["Match"] = df["match_acc"] * 100.0
    df["F1"] = df["f1"] * 100.0

    # Identify question types in preferred order
    q_order = _ordered_qtypes(sorted(df["question_type"].dropna().unique().tolist()))

    # Build a wide table per agent with two columns (Match, F1) for each qtype
    # First, reshape so we can pivot both metrics
    melted = df.melt(
        id_vars=["agent_id", "question_type"],
        value_vars=["Match", "F1"],
        var_name="metric",
        value_name="value",
    )

    wide = (
        melted.pivot_table(
            index="agent_id",
            columns=["question_type", "metric"],
            values="value",
            aggfunc="mean",
        )
        .reindex(columns=pd.MultiIndex.from_product([q_order, ["Match", "F1"]]))
        .sort_index()
    )

    # Convert to a regular DataFrame with Model/Obj. up front
    model_obj_rows = [ _parse_agent_to_model_obj(aid, agent_map) for aid in wide.index ]
    model_col = [m for (m, _) in model_obj_rows]
    obj_col = [o for (_, o) in model_obj_rows]

    # Format as 2 decimal strings
    formatted = wide.applymap(lambda x: f"{x:.2f}" if pd.notnull(x) else "")

    # Build final frame
    formatted.insert(0, ("_", "Obj."), obj_col)     # temporary prefix for stable order
    formatted.insert(0, ("_", "Model"), model_col)
    formatted.columns = pd.MultiIndex.from_tuples(formatted.columns)

    # Sort columns: Model, Obj., then qtypes x (Match,F1)
    cols = [("_", "Model"), ("_", "Obj.")] + [(q, sub) for q in q_order for sub in ("Match", "F1")]
    formatted = formatted.reindex(columns=pd.MultiIndex.from_tuples(cols))

    # Remove the helper top-level "_" for the first two columns by flattening afterwards
    formatted.columns = pd.MultiIndex.from_tuples(
        [("Model", "") if c == ("_", "Model") else
         ("Obj.", "") if c == ("_", "Obj.") else c for c in formatted.columns]
    )

    # Reset index to turn agent_id into rows (will be replaced by Model/Obj.)
    formatted = formatted.reset_index(drop=True)

    return formatted


def save_latex_table(df_latex: pd.DataFrame, path: str):
    # Use to_latex with MultiIndex columns, no escaping (we aren't adding LaTeX chars here)
    with open(path, "w", encoding="utf-8") as f:
        f.write(df_latex.to_latex(index=False, escape=False, multicolumn=True, multicolumn_format='c'))

#### Test run

In [8]:
RESULTS_PATH = "results/results__test_run_001.jsonl"
OUTDIR = "output/figures/test_001"
LATEX_TABLE_PATH = "output/tables/"

In [16]:
configure_logging()

rows = load_jsonl(RESULTS_PATH)

df = compute_metrics_df(rows)
agg = aggregate_by_agent_qtype(df)

In [17]:
os.makedirs(OUTDIR, exist_ok=True)
df.to_csv(os.path.join(OUTDIR, "per_example_with_metrics.csv"), index=False)
agg.to_csv(os.path.join(OUTDIR, "agg_by_agent_qtype.csv"), index=False)

make_plots(agg, OUTDIR)

In [20]:
# LaTeX export
agent_map = None

latex_df = build_latex_table_df(agg, agent_map=agent_map)
latex_df.to_csv(os.path.join(LATEX_TABLE_PATH, "latex_table_preview.csv"), index=False)

os.makedirs(os.path.dirname(LATEX_TABLE_PATH), exist_ok=True)
save_latex_table(latex_df, os.path.join(LATEX_TABLE_PATH, "latex_table.tex"))

logging.info(f"Analysis complete. Outputs in: {OUTDIR}")
if LATEX_TABLE_PATH:
    logging.info(f"LaTeX table saved to: {LATEX_TABLE_PATH}")


/var/folders/6p/s0rnd_zn3zvbjzlxtjryt_t80000gn/T/ipykernel_1865/4070005341.py:265: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted = wide.applymap(lambda x: f"{x:.2f}" if pd.notnull(x) else "")
2025-08-22 00:32:42,374 | INFO | Analysis complete. Outputs in: output/figures/test_001
2025-08-22 00:32:42,374 | INFO | LaTeX table saved to: output/tables/


In [ ]:
def main():
    configure_logging()

    rows = load_jsonl(RESULTS_PATH)
    if not rows:
        logging.error(f"No rows loaded from {RESULTS_PATH}. Exiting.")
        return

    df = compute_metrics_df(rows)
    agg = aggregate_by_agent_qtype(df)

    os.makedirs(OUTDIR, exist_ok=True)
    # Save raw with metrics
    df.to_csv(os.path.join(OUTDIR, "per_example_with_metrics.csv"), index=False)
    # Save aggregation
    agg.to_csv(os.path.join(OUTDIR, "agg_by_agent_qtype.csv"), index=False)

    # Plots
    make_plots(agg, OUTDIR)

    # LaTeX export
    agent_map = None

    latex_df = build_latex_table_df(agg, agent_map=agent_map)
    latex_df.to_csv(os.path.join(OUTDIR, "latex_table_preview.csv"), index=False)

    if LATEX_TABLE_PATH:
        os.makedirs(os.path.dirname(LATEX_TABLE_PATH) or ".", exist_ok=True)
        save_latex_table(latex_df, LATEX_TABLE_PATH)

    logging.info(f"Analysis complete. Outputs in: {OUTDIR}")
    if LATEX_TABLE_PATH:
        logging.info(f"LaTeX table saved to: {LATEX_TABLE_PATH}")


if __name__ == "__main__":
    main()
